In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Subset, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Normalisation standardisée sur [-1, 1]
transform_1d = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
transform_3d = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# ---------------------------------------------------------
# FASHION-MNIST (Exemple avec 30 000 images au lieu de 60 000)
# ---------------------------------------------------------
fmnist_full = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform_1d)

# On sélectionne uniquement les 30 000 premiers indices
fmnist_subset = Subset(fmnist_full, range(30000))

# Séparation sur ce nouveau sous-ensemble (ex: 25k train, 5k validation)
fmnist_train, fmnist_val = random_split(fmnist_subset, [25000, 5000])

fmnist_trainloader = torch.utils.data.DataLoader(fmnist_train, batch_size=512, shuffle=True)
fmnist_valloader = torch.utils.data.DataLoader(fmnist_val, batch_size=512, shuffle=False)

# ---------------------------------------------------------
# CIFAR-10 (Limité à 30 000 images au lieu de 50 000)
# ---------------------------------------------------------
cifar_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_3d)

# On sélectionne uniquement les 30 000 premiers indices
cifar_subset = Subset(cifar_full, range(30000))

# Séparation sur ce nouveau sous-ensemble (ex: 25k train, 5k validation)
cifar_train, cifar_val = random_split(cifar_subset, [25000, 5000])

cifar_trainloader = torch.utils.data.DataLoader(cifar_train, batch_size=256, shuffle=True)
cifar_valloader = torch.utils.data.DataLoader(cifar_val, batch_size=256, shuffle=False)

# ---------------------------------------------------------
# DICTIONNAIRE DES DATASETS
# ---------------------------------------------------------
datasets = {
    "fMNIST": {"train": fmnist_trainloader, "val": fmnist_valloader, "input_dim": 784, "shape": (1, 28, 28)},
    "CIFAR10": {"train": cifar_trainloader, "val": cifar_valloader, "input_dim": 3072, "shape": (3, 32, 32)}
}

print("Datasets chargés et réduits avec succès !")
#print(f"Taille de l'entraînement CIFAR-10 : {len(cifar_train)} images")
#print(f"Taille de la validation CIFAR-10  : {len(cifar_val)} images")

Datasets chargés et réduits avec succès !


In [2]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader

# LE DÉTAIL QUI SAUVE L'ARCHITECTURE : transforms.Pad(2)
transform_fmnist_padded = transforms.Compose([
    transforms.Pad(2), # Ajoute 2 pixels de bordure (28x28 -> 32x32)
    transforms.ToTensor(), 
    transforms.Normalize((0.5,), (0.5,))
])

print("Rechargement de Fashion-MNIST avec Padding (32x32)...")

# Application de la nouvelle transformation
fmnist_full_padded = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform_fmnist_padded)

# On refait la séparation Train/Val (ou Subset si tu avais limité à 30k)
fmnist_train, fmnist_val = random_split(fmnist_full_padded, [50000, 10000])

# Mise à jour stricte de ton dictionnaire de datasets
datasets["fMNIST"]["train"] = DataLoader(fmnist_train, batch_size=1024, shuffle=True)
datasets["fMNIST"]["val"] = DataLoader(fmnist_val, batch_size=1024, shuffle=False)

# Vérification
sample_images, _ = next(iter(datasets["fMNIST"]["train"]))
print(f"Nouvelle taille des images : {sample_images.shape} (Attendu: Batch x 1 x 32 x 32)")

Rechargement de Fashion-MNIST avec Padding (32x32)...
Nouvelle taille des images : torch.Size([1024, 1, 32, 32]) (Attendu: Batch x 1 x 32 x 32)


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

class VGG5_bPC_Paper(nn.Module):
    def __init__(self, num_labels=10, rep_neurons=256, alpha_gen=1e-4, alpha_disc=1.0):
        super().__init__()
        self.L = 6
        self.alpha_gen = alpha_gen
        self.alpha_disc = alpha_disc
        self.latent_dim = num_labels + rep_neurons # 266 neurones
        self.num_labels = num_labels
        
        self.activation = nn.LeakyReLU()

        # ---------------------------------------------------------
        # VOIE DISCRIMINATIVE (V) : Conv (Stride 1) + MaxPool
        # ---------------------------------------------------------
        self.V_convs = nn.ModuleList([
            nn.Sequential(nn.Conv2d(1, 128, kernel_size=3, stride=1, padding=1), nn.MaxPool2d(2, 2)),
            nn.Sequential(nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1), nn.MaxPool2d(2, 2)),
            nn.Sequential(nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1), nn.MaxPool2d(2, 2)),
            nn.Sequential(nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1), nn.MaxPool2d(2, 2))
        ])
        # Sortie du Flatten pour 32x32 d'entrée : 512 * 2 * 2 = 2048
        # L'activation vers x_L est une identité
        self.V_linear = nn.Sequential(nn.Flatten(), nn.Linear(2048, self.latent_dim), nn.Identity()) 

        # ---------------------------------------------------------
        # VOIE GÉNÉRATIVE (W) : ConvTranspose (Stride 2)[cite: 2]
        # ---------------------------------------------------------
        self.W_linear = nn.Sequential(nn.Linear(self.latent_dim, 2048), nn.Unflatten(1, (512, 2, 2)))
        
        self.W_convs = nn.ModuleList([
            nn.ConvTranspose2d(512, 512, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ConvTranspose2d(512, 256, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            # La prédiction vers x_1 utilise tanh pour forcer la sortie dans [-1, 1][cite: 2]
            nn.Sequential(nn.ConvTranspose2d(128, 1, kernel_size=3, stride=2, padding=1, output_padding=1), nn.Tanh())
        ])

        # Pondération des énergies : alpha_disc des neurones libres est défini sur alpha_gen[cite: 2]
        alpha_disc_L = torch.ones(self.latent_dim) * alpha_disc
        alpha_disc_L[num_labels:] = alpha_gen 
        self.register_buffer('alpha_disc_L', alpha_disc_L)

    def _forward_V(self, x, layer_idx):
        if layer_idx < 4: return self.V_convs[layer_idx](self.activation(x))
        return self.V_linear(self.activation(x))

    def _forward_W(self, x, layer_idx):
        if layer_idx == 4: return self.W_linear(self.activation(x))
        return self.W_convs[3 - layer_idx](self.activation(x))

    def compute_energy(self, x):
        energy_gen = 0.0
        energy_disc = 0.0
        
        for i in range(self.L - 1):
            pred_gen = self._forward_W(x[i+1], i)
            # L'énergie standard bPC (l'article utilise la distance L2 classique)[cite: 2]
            energy_gen += torch.sum((x[i] - pred_gen) ** 2) * (self.alpha_gen / 2)
            
            pred_disc = self._forward_V(x[i], i)
            diff_sq_disc = (x[i+1] - pred_disc) ** 2
            
            if i == self.L - 2:
                # Applique la pondération spécifique à la couche latente (alpha_gen pour les neurones libres)[cite: 2]
                energy_disc += torch.sum(diff_sq_disc * self.alpha_disc_L) / 2
            else:
                energy_disc += torch.sum(diff_sq_disc) * (self.alpha_disc / 2)
                
        return energy_gen + energy_disc

    def bottom_up_sweep(self, x1):
        x = [x1.clone()]
        with torch.no_grad():
            for i in range(self.L - 1):
                x.append(self._forward_V(x[-1], i))
        return x

    def infer(self, x_init, clamped_indices, steps=32, lr_x=0.01, partial_clamp=None, activity_decay=0.0):
        x = [tensor.clone() for tensor in x_init]
        free_params = []
        for i in range(self.L):
            if i not in clamped_indices:
                x[i].requires_grad = True
                free_params.append(x[i])
                
        if len(free_params) > 0:
            optimizer_x = optim.SGD(free_params, lr=lr_x, momentum=0.0)
            for _ in range(steps):
                optimizer_x.zero_grad()
                energy = self.compute_energy(x)
                energy.backward()
                
                # Un "activity decay" est appliqué à la sous-population de représentation dans x_L pour régulariser l'apprentissage[cite: 2]
                if activity_decay > 0.0 and x[-1].requires_grad:
                    x[-1].grad.data[:, self.num_labels:] += activity_decay * x[-1].data[:, self.num_labels:]
                
                if partial_clamp is not None:
                    layer_idx, mask = partial_clamp
                    if x[layer_idx].requires_grad:
                        x[layer_idx].grad.data *= (1.0 - mask)
                        
                optimizer_x.step()
                
        return [tensor.detach() for tensor in x]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instanciation avec 266 neurones (10 + 256)[cite: 2]
bpc_model = VGG5_bPC_Paper(num_labels=10, rep_neurons=256, alpha_gen=1e-4, alpha_disc=1.0).to(device)

optimizer_theta = optim.AdamW(bpc_model.parameters(), lr=1e-4, weight_decay=1e-4)

# Masque pour le clampage partiel : on fige uniquement les 10 premières composantes
latent_mask = torch.zeros(266, device=device)
latent_mask[:10] = 1.0  

epochs = 25 
print("Début de l'entraînement bPC (Algorithme exact du papier)...")

for epoch in range(epochs):
    bpc_model.train()
    total_energy = 0.0
    
    for images, labels in datasets["fMNIST"]["train"]:
        x1_input = images.to(device)
        batch_size = x1_input.size(0)
        
        # Le tenseur latent complet (266) : one-hot pour les 10 premiers, 0 pour les 256 libres
        xL_label = torch.zeros((batch_size, 266), device=device)
        xL_label[:, :10] = torch.nn.functional.one_hot(labels, num_classes=10).float()
        
        # 1. Initialisation par balayage ascendant (bottom-up sweep)[cite: 2]
        x_init = bpc_model.bottom_up_sweep(x1_input)
        x_init[0] = x1_input
        x_init[-1][:, :10] = xL_label[:, :10]
        
        # 2. Inférence itérative avec T=32[cite: 2]
        # Un 'activity_decay' est appliqué à la sous-population de représentation[cite: 2]
        x_inferred = bpc_model.infer(
            x_init, 
            clamped_indices=[0], 
            steps=32, 
            lr_x=0.01, 
            partial_clamp=(5, latent_mask.unsqueeze(0)),
            activity_decay=1e-3 
        )
        
        # 3. Mise à jour des poids avec la nouvelle configuration
        optimizer_theta.zero_grad()
        loss = bpc_model.compute_energy(x_inferred)
        loss.backward()
        optimizer_theta.step()
        
        total_energy += loss.item()
    print(loss)

Début de l'entraînement bPC (Algorithme exact du papier)...
tensor(280.3703, device='cuda:0', grad_fn=<AddBackward0>)
tensor(238.0855, device='cuda:0', grad_fn=<AddBackward0>)
tensor(214.3424, device='cuda:0', grad_fn=<AddBackward0>)
tensor(195.8471, device='cuda:0', grad_fn=<AddBackward0>)
tensor(179.1928, device='cuda:0', grad_fn=<AddBackward0>)
tensor(193.0436, device='cuda:0', grad_fn=<AddBackward0>)
tensor(177.9573, device='cuda:0', grad_fn=<AddBackward0>)
tensor(161.0515, device='cuda:0', grad_fn=<AddBackward0>)
tensor(161.9110, device='cuda:0', grad_fn=<AddBackward0>)
